# Full Pipeline

### WORLD CLOCK — Full Pipeline

 Input:  a string the user typed (abbreviation, city, raw IANA zone)
 Output: a canonical IANA zone + current local time

 Layers, in the order they fire at runtime:
   1. Abbreviation resolution   (JST -> Asia/Tokyo)
   2. Place name lookup         (Paris -> Europe/Paris)
   3. IANA validation           (is this a real zone?)
   4. Offset + DST              (compute local time)
   5. Alias canonicalization    (Asia/Calcutta -> Asia/Kolkata)

Layers 3, 4, 5 are pure computation from the tz database.
Layers 1 and 2 need data: ABBREV_MAP (built at startup), and GeoNames.

In [10]:
import json
import os
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import zoneinfo

# Optional: coord -> zone lookup, for future use
try:
    from timezonefinder import TimezoneFinder
    _TF = TimezoneFinder()
except ImportError:
    _TF = None


In [11]:
# ---------------------------------------------------------------------------
# LAYER 5 — Alias resolution
# ---------------------------------------------------------------------------
# Load aliases.json produced by build_aliases.py.
# If it's missing, canonicalize() degrades to identity, which is safe
# (unknown names pass through unchanged).

ALIASES = {}
_alias_path = Path("aliases.json")
if _alias_path.exists():
    with _alias_path.open(encoding="utf-8") as f:
        ALIASES = json.load(f)
    print(f"[Layer 5] loaded {len(ALIASES)} aliases from {_alias_path}")
else:
    print("[Layer 5] WARNING: aliases.json not found — canonicalize() is identity")


def canonicalize(zone_name: str) -> str:
    """Return the canonical IANA name for any zone string. Idempotent."""
    if not zone_name:
        return zone_name
    return ALIASES.get(zone_name, zone_name)

[Layer 5] loaded 257 aliases from aliases.json


In [12]:
# ---------------------------------------------------------------------------
# LAYER 3 — IANA validation
# ---------------------------------------------------------------------------

def is_valid_zone(name: str) -> bool:
    """True if `name` resolves to a loadable TZif file on this machine."""
    if not isinstance(name, str) or not name:
        return False
    try:
        ZoneInfo(name)
        return True
    except (ZoneInfoNotFoundError, ValueError, OSError):
        return False

In [13]:
# ---------------------------------------------------------------------------
# LAYER 4 — Offset, DST, abbreviation, local time
# ---------------------------------------------------------------------------

def fmt_offset(delta):
    """Format a timedelta as +HH:MM or -HH:MM."""
    if delta is None:
        return "n/a"
    total = int(round(delta.total_seconds()))
    sign = "+" if total >= 0 else "-"
    total = abs(total)
    hh, rem = divmod(total, 3600)
    mm, ss = divmod(rem, 60)
    if ss:
        return f"{sign}{hh:02d}:{mm:02d}:{ss:02d}"
    return f"{sign}{hh:02d}:{mm:02d}"


def snapshot(zone_name: str, moment: datetime | None = None) -> dict:
    """Everything Layer 4 needs about one zone at one instant."""
    tz = ZoneInfo(zone_name)
    if moment is None:
        moment = datetime.now(timezone.utc)
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=timezone.utc)

    instant_utc = moment.astimezone(timezone.utc)
    local = instant_utc.astimezone(tz)

    offset = local.utcoffset()
    dst_delta = local.dst() or __import__("datetime").timedelta(0)
    std_offset = offset - dst_delta

    return {
        "zone": zone_name,
        "local": local,
        "utc": instant_utc,
        "abbrev": local.tzname(),
        "offset": offset,
        "offset_str": fmt_offset(offset),
        "std_offset": std_offset,
        "std_offset_str": fmt_offset(std_offset),
        "dst_delta": dst_delta,
        "dst_active": dst_delta != __import__("datetime").timedelta(0),
    }


def convert_time(local_str: str, from_zone: str, to_zone: str) -> dict:
    """Convert a naive local time string from one zone to another."""
    naive = datetime.fromisoformat(local_str)
    src = ZoneInfo(canonicalize(from_zone))
    dst = ZoneInfo(canonicalize(to_zone))
    aware = naive.replace(tzinfo=src)
    converted = aware.astimezone(dst)
    return {
        "source_local": aware,
        "source_offset": aware.utcoffset(),
        "source_abbrev": aware.tzname(),
        "source_utc": aware.astimezone(timezone.utc),
        "target_local": converted,
        "target_offset": converted.utcoffset(),
        "target_abbrev": converted.tzname(),
    }


In [14]:
# ---------------------------------------------------------------------------
# LAYER 2 — Place name lookup (GeoNames)
# ---------------------------------------------------------------------------

def load_places(path: str):
    """Load GeoNames TSV into a name -> list of place dicts index."""
    index = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 18:
                continue
            try:
                place = {
                    "name": fields[1],
                    "ascii_name": fields[2],
                    "lat": float(fields[4]),
                    "lon": float(fields[5]),
                    "country": fields[8],
                    "population": int(fields[14]) if fields[14] else 0,
                    "zone": fields[17] or None,
                }
            except (ValueError, IndexError):
                continue

            # Index by primary name, ascii name, and alternate names
            keys = {place["name"].lower(), place["ascii_name"].lower()}
            for alt in (fields[3] or "").split(","):
                alt = alt.strip().lower()
                if alt:
                    keys.add(alt)
            for key in keys:
                index[key].append(place)
    return index


# Try small file first, fall back to big one
_PLACES = None
for candidate in ["geonames/cities15000.txt", "geonames/cities5000.txt",
                  "geonames/allCountries.txt"]:
    if Path(candidate).exists():
        print(f"[Layer 2] loading {candidate} ...")
        _PLACES = load_places(candidate)
        print(f"[Layer 2] loaded {len(_PLACES):,} unique keys")
        break
if _PLACES is None:
    print("[Layer 2] WARNING: no GeoNames file found")


def lookup_place(name: str) -> list:
    """Return all places matching `name`, sorted by population desc."""
    if not _PLACES:
        return []
    matches = _PLACES.get(name.lower(), [])
    return sorted(matches, key=lambda p: p["population"], reverse=True)


def top_zone(name: str) -> str | None:
    """Return the IANA zone for the most populous place matching `name`."""
    results = lookup_place(name)
    if not results:
        return None
    return results[0]["zone"]

[Layer 2] loading geonames/cities15000.txt ...
[Layer 2] loaded 346,886 unique keys


In [15]:
# ---------------------------------------------------------------------------
# LAYER 1 — Abbreviation resolution
# ---------------------------------------------------------------------------
# Build the reverse map from abbreviation -> canonical zones.
# Uses canonicalize() so aliases collapse and JST becomes unique.
# This is expensive (iterates every zone), so build it once at startup.

def build_abbrev_map() -> dict:
    now_utc = datetime.now(timezone.utc)
    raw = defaultdict(set)
    for zone_name in zoneinfo.available_timezones():
        try:
            tz = zoneinfo.ZoneInfo(zone_name)
            local = now_utc.astimezone(tz)
            abbrev = local.tzname()
            if abbrev and abbrev.isalpha() and abbrev.isupper():
                raw[abbrev].add(canonicalize(zone_name))
        except Exception:
            continue
    return {abbr: sorted(zones) for abbr, zones in raw.items()}


ABBREV_MAP = build_abbrev_map()
print(f"[Layer 1] built abbreviation map: {len(ABBREV_MAP)} abbreviations")


def resolve_abbreviation(abbr: str):
    """Return (status, zones) where status is unique | ambiguous | unknown."""
    a = abbr.strip().upper()
    zones = ABBREV_MAP.get(a, [])
    if not zones:
        return "unknown", []
    if len(zones) == 1:
        return "unique", zones
    return "ambiguous", zones


[Layer 1] built abbreviation map: 43 abbreviations


In [16]:
# ---------------------------------------------------------------------------
# THE PIPELINE — one entry point that tries every layer in order
# ---------------------------------------------------------------------------

def resolve_user_input(query: str) -> dict:
    """
    Given anything the user typed, return a resolution dict.

    Resolution order:
      1. Abbreviation     (JST, IST)              -> Layer 1
      2. Place name       (Paris, New York)       -> Layer 2
      3. Raw IANA zone    (America/New_York)      -> Layer 5 canonicalize
      4. Coordinate       ("48.85,2.35")          -> Layer 2b (optional)

    Returns:
        {"status": "unique" | "ambiguous" | "unknown",
         "zone": str | None,
         "candidates": list[str],
         "source": str}
    """
    q = query.strip()
    if not q:
        return {"status": "unknown", "zone": None,
                "candidates": [], "source": "none"}

    # --- 1. Abbreviation ---
    status, zones = resolve_abbreviation(q)
    if status == "unique":
        return {"status": "unique", "zone": zones[0],
                "candidates": zones, "source": "abbreviation"}
    if status == "ambiguous":
        return {"status": "ambiguous", "zone": None,
                "candidates": zones, "source": "abbreviation"}

    # --- 2. Coordinate ("lat,lon") ---
    if _TF is not None and "," in q:
        try:
            lat_s, lon_s = q.split(",", 1)
            lat, lon = float(lat_s.strip()), float(lon_s.strip())
            zone = _TF.timezone_at(lat=lat, lng=lon)
            if zone:
                canon = canonicalize(zone)
                return {"status": "unique", "zone": canon,
                        "candidates": [canon], "source": "coordinate"}
        except (ValueError, AttributeError):
            pass  # fall through

    # --- 3. Place name ---
    zone = top_zone(q)
    if zone:
        canon = canonicalize(zone)
        return {"status": "unique", "zone": canon,
                "candidates": [canon], "source": "place"}

    # --- 4. Raw IANA zone name ---
    if is_valid_zone(q):
        canon = canonicalize(q)
        return {"status": "unique", "zone": canon,
                "candidates": [canon], "source": "zone"}

    return {"status": "unknown", "zone": None,
            "candidates": [], "source": "none"}



In [17]:
# ---------------------------------------------------------------------------
# DEMO
# ---------------------------------------------------------------------------

def show(query: str):
    r = resolve_user_input(query)
    if r["status"] == "unique":
        s = snapshot(r["zone"])
        print(f"  {query:<18} -> {r['zone']:<22} "
              f"{s['local']:%H:%M %Z (%z)}  [{r['source']}]")
    elif r["status"] == "ambiguous":
        print(f"  {query:<18} -> AMBIGUOUS: {len(r['candidates'])} candidates")
        for c in r["candidates"][:5]:
            s = snapshot(c)
            print(f"      {c:<22} {s['local']:%H:%M %Z (%z)}")
        if len(r["candidates"]) > 5:
            print(f"      ... and {len(r['candidates']) - 5} more")
    else:
        print(f"  {query:<18} -> UNKNOWN")


if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("FULL PIPELINE DEMO")
    print("=" * 70)

    for q in [
        "JST",                    # abbreviation, unique
        "IST",                    # abbreviation, ambiguous
        "GMT",                    # abbreviation, ambiguous
        "Paris",                  # place
        "New York",               # place (via alternatenames)
        "Tokyo",                  # place
        "Asia/Calcutta",          # alias -> canonical
        "US/Eastern",             # backward link
        "America/New_York",       # raw zone
        "48.85,2.35",             # coordinate (Paris)
        "xyz123",                 # unknown
    ]:
        show(q)

    print("\n" + "=" * 70)
    print("CONVERSIONS THROUGH THE PIPELINE")
    print("=" * 70)

    # End-to-end: name -> zone -> convert to another name -> zone
    for src, dst, when in [
        ("New York", "Tokyo", "2026-03-21 09:00"),
        ("Paris", "Mumbai", "2026-06-15 12:00"),
    ]:
        src_zone = resolve_user_input(src)["zone"]
        dst_zone = resolve_user_input(dst)["zone"]
        if src_zone and dst_zone:
            r = convert_time(when, src_zone, dst_zone)
            print(f"\n  {when}  {src} ({src_zone})  ->  {dst} ({dst_zone})")
            print(f"      source: {r['source_local']:%Y-%m-%d %H:%M:%S} "
                  f"{fmt_offset(r['source_offset'])} {r['source_abbrev']}")
            print(f"      target: {r['target_local']:%Y-%m-%d %H:%M:%S} "
                  f"{fmt_offset(r['target_offset'])} {r['target_abbrev']}")


FULL PIPELINE DEMO
  JST                -> Asia/Tokyo             18:37 JST (+0900)  [abbreviation]
  IST                -> AMBIGUOUS: 2 candidates
      Asia/Kolkata           15:07 IST (+0530)
      Europe/Dublin          10:37 IST (+0100)
  GMT                -> AMBIGUOUS: 6 candidates
      Africa/Abidjan         09:37 GMT (+0000)
      Africa/Bissau          09:37 GMT (+0000)
      Africa/Monrovia        09:37 GMT (+0000)
      Africa/Sao_Tome        09:37 GMT (+0000)
      America/Danmarkshavn   09:37 GMT (+0000)
      ... and 1 more
  Paris              -> Europe/Paris           11:37 CEST (+0200)  [place]
  New York           -> America/New_York       05:37 EDT (-0400)  [place]
  Tokyo              -> Asia/Tokyo             18:37 JST (+0900)  [place]
  Asia/Calcutta      -> Asia/Kolkata           15:07 IST (+0530)  [zone]
  US/Eastern         -> America/New_York       05:37 EDT (-0400)  [zone]
  America/New_York   -> America/New_York       05:37 EDT (-0400)  [zone]
  48.85,2.3